# concateFile.ipynb

Notebook for advanced ECG concatenation with sequential HR ordering, pure/mixed level segments, and duration-aware truncation.

## Requirements implemented
- Sequential HR ordering (ascending HR number)
- Pure level segments (single label)
- Mixed level segments with majority-vote labels (ties -> lowest label)
- Duration-aware concatenation with smart truncation
- Time continuity preservation and column cleanup

In [2]:
import hashlib
import os
import random 
import re
from collections import Counter
from typing import Dict, List, Optional, Tuple

import pandas as pd

In [3]:
class ECGAdvancedConcatenator:
    """
    Advanced ECG data concatenator with sequential HR ordering and duration-aware truncation.
    """

    def __init__(self, csv_label_file: Optional[str], data_dir: str, labels: Optional[List[int]] = None) -> None:
        if csv_label_file is not None and not os.path.isfile(csv_label_file):
            raise FileNotFoundError(f"CSV label file not found: {csv_label_file}")
        if not os.path.isdir(data_dir):
            raise FileNotFoundError(f"Data directory not found: {data_dir}")

        self.csv_label_file = csv_label_file
        self.data_dir = data_dir
        self.labels = labels or [0, 1, 2, 3]
        self.label_files: Dict[int, List[str]] = {}
        self.data_cache: Dict[str, pd.DataFrame] = {}

        if self.csv_label_file:
            self._load_label_mapping()
        else:
            self._scan_label_directories()

    def _scan_label_directories(self) -> None:
        for label in self.labels:
            label_dir = os.path.join(self.data_dir, str(label))
            if not os.path.isdir(label_dir):
                raise FileNotFoundError(f"Label directory not found: {label_dir}")
            files = [f for f in os.listdir(label_dir) if f.lower().endswith(".csv")]
            if not files:
                raise FileNotFoundError(f"No CSV files found in {label_dir}")
            self.label_files[label] = sorted(files, key=self._extract_hr_number)

    def _load_label_mapping(self) -> None:
        df = pd.read_csv(self.csv_label_file)
        df.columns = [c.strip() for c in df.columns]
        if "File" not in df.columns or "Label" not in df.columns:
            raise ValueError("CSV must include 'File' and 'Label' columns.")

        for _, row in df.iterrows():
            label = int(row["Label"])
            filename = str(row["File"])
            self.label_files.setdefault(label, []).append(filename)

        for label in self.label_files:
            self.label_files[label] = sorted(self.label_files[label], key=self._extract_hr_number)

    def _get_full_path(self, label: int, filename: str) -> str:
        return os.path.join(self.data_dir, str(label), filename)

    def _load_label_files(self, label: int) -> None:
        if label not in self.label_files:
            raise ValueError(f"Label {label} is not available.")

        for filename in self.label_files[label]:
            full_path = self._get_full_path(label, filename)
            if full_path in self.data_cache:
                continue
            if not os.path.isfile(full_path):
                raise FileNotFoundError(f"File missing for label {label}: {full_path}")
            df = pd.read_csv(full_path)
            df.columns = [c.strip() for c in df.columns]
            self.data_cache[full_path] = df

    @staticmethod
    def _get_duration_from_dataframe(df: pd.DataFrame) -> float:
        if "Time" in df.columns:
            time_series = df["Time"].to_numpy()
            if len(time_series) == 0:
                return 0.0
            return float(time_series[-1] - time_series[0])
        return float(len(df))

    @staticmethod
    def _extract_hr_number(filename: str) -> int:
        base = os.path.splitext(os.path.basename(filename))[0].lower()
        # Support names like hr80.csv and hr80_1.csv by reading the first hr<number> token.
        m = re.search(r"hr(\d+)", base)
        if m:
            return int(m.group(1))

        # Fallback: only parse the first numeric token before '_' to avoid hr80_1 -> 801.
        first_token = base.split("_", 1)[0]
        m2 = re.search(r"(\d+)", first_token)
        if m2:
            return int(m2.group(1))

        raise ValueError(f"Unable to extract HR number from filename: {filename}")

    @staticmethod
    def _offset_time(df: pd.DataFrame, offset: float) -> pd.DataFrame:
        if "Time" not in df.columns:
            return df
        df = df.copy()
        df["Time"] = df["Time"] + offset
        return df

    @staticmethod
    def _get_next_file_index(directory: str, prefix: str) -> int:
        max_index = 0
        if os.path.isdir(directory):
            for name in os.listdir(directory):
                if not name.lower().endswith(".csv"):
                    continue
                stem = os.path.splitext(name)[0]
                token = f"{prefix}_"
                if not stem.startswith(token):
                    continue
                suffix = stem[len(token):]
                if suffix.isdigit():
                    max_index = max(max_index, int(suffix))
        return max_index + 1

    @staticmethod
    def _hash_dataframe(df: pd.DataFrame) -> str:
        csv_bytes = df.to_csv(index=False).encode("utf-8")
        return hashlib.sha256(csv_bytes).hexdigest()

    @staticmethod
    def _collect_hashes_in_folder(folder: str) -> set:
        hashes = set()
        if not os.path.isdir(folder):
            return hashes
        for name in os.listdir(folder):
            if not name.lower().endswith(".csv"):
                continue
            fp = os.path.join(folder, name)
            with open(fp, "rb") as fh:
                hashes.add(hashlib.sha256(fh.read()).hexdigest())
        return hashes

    def concatenate_preserve_time(self, label: int, duration_minutes: float, random_order: bool = True) -> pd.DataFrame:
        self._load_label_files(label)
        label_files = list(self.label_files[label])
        if random_order:
            random.shuffle(label_files)
        else:
            label_files = sorted(label_files, key=self._extract_hr_number)

        target_seconds = duration_minutes * 60.0
        total_duration = 0.0
        output_parts = []
        time_offset = 0.0

        for filename in label_files:
            full_path = self._get_full_path(label, filename)
            df = self.data_cache[full_path]
            duration = self._get_duration_from_dataframe(df)

            remaining = target_seconds - total_duration
            if remaining <= 0:
                break

            if duration <= remaining:
                output_parts.append(self._offset_time(df, time_offset))
                total_duration += duration
                if "Time" in df.columns and len(df) > 0:
                    time_offset = output_parts[-1]["Time"].iloc[-1]
                continue

            # Truncate last file to match the remaining duration.
            truncated = df.copy()
            if "Time" in truncated.columns:
                start_time = truncated["Time"].iloc[0]
                cutoff = start_time + remaining
                truncated = truncated[truncated["Time"] <= cutoff]
                if len(truncated) > 0:
                    truncated["Time"] = truncated["Time"] - start_time + time_offset
            else:
                truncated = truncated.iloc[: int(remaining)]
            output_parts.append(truncated)
            total_duration = target_seconds
            break

        if not output_parts:
            raise ValueError(f"No data available for label {label}.")

        return pd.concat(output_parts, ignore_index=True)

    def concatenate_sequential_hr(self, labels: List[int], num_segments: int, output_dir: str, target_minutes: float) -> None:
        if not labels:
            raise ValueError("Labels list cannot be empty.")
        if num_segments <= 0:
            raise ValueError("num_segments must be positive.")

        os.makedirs(output_dir, exist_ok=True)

        labeled_files: List[Tuple[int, str]] = []
        for label in labels:
            self._load_label_files(label)
            for filename in self.label_files[label]:
                labeled_files.append((label, filename))

        labeled_files.sort(key=lambda item: (self._extract_hr_number(item[1]), item[0]))

        target_seconds = target_minutes * 60.0
        segments: List[Tuple[int, str]] = []

        for label, filename in labeled_files:
            segments.append((label, filename))
            if len(segments) < num_segments:
                continue

            output_df, majority_label = self._build_duration_segment(segments, target_seconds)
            if len(labels) > 1:
                final_dir = os.path.join(output_dir, f"mixed_{majority_label}")
                file_prefix = f"mixed_{majority_label}"
            else:
                final_dir = output_dir
                file_prefix = f"concat_{majority_label}"
            os.makedirs(final_dir, exist_ok=True)
            next_index = self._get_next_file_index(final_dir, file_prefix)
            file_name = f"{file_prefix}_{next_index:03d}.csv"

            output_path = os.path.join(final_dir, file_name)
            output_df.to_csv(output_path, index=False)

            segments = []

    def concatenate_random_hr(
        self,
        labels: List[int],
        files_per_segment: int,
        n_outputs: int,
        output_dir: str,
        target_minutes: float,
        allow_replacement: bool = True,
        random_seed: Optional[int] = None,
        files_per_segment_max: Optional[int] = None,
        required_majority_label: Optional[int] = None,
        min_majority_ratio: float = 0.50,
        hr_band_by_label: Optional[Dict[int, Tuple[int, int]]] = None,
        choose_unique_hr_variants: bool = True,
        adaptive_relaxation: bool = True,
    ) -> None:
        if not labels:
            raise ValueError("Labels list cannot be empty.")
        if files_per_segment <= 0:
            raise ValueError("files_per_segment must be positive.")
        if files_per_segment_max is None:
            files_per_segment_max = files_per_segment
        if files_per_segment_max < files_per_segment:
            raise ValueError("files_per_segment_max must be >= files_per_segment.")
        if n_outputs <= 0:
            raise ValueError("n_outputs must be positive.")

        rng = random.Random(random_seed)
        os.makedirs(output_dir, exist_ok=True)

        labeled_files: List[Tuple[int, str]] = []
        for label in labels:
            self._load_label_files(label)
            for filename in self.label_files[label]:
                labeled_files.append((label, filename))

        if not labeled_files:
            raise ValueError("No files available for selected labels.")

        # Group hrX/hrX_1... variants and sample one file per HR key for better diversity.
        hr_group_pool: List[Tuple[int, int, List[str]]] = []
        if choose_unique_hr_variants:
            grouped: Dict[Tuple[int, int], List[str]] = {}
            for label, filename in labeled_files:
                hr_num = self._extract_hr_number(filename)
                grouped.setdefault((label, hr_num), []).append(filename)
            hr_group_pool = [
                (label, hr_num, sorted(variants))
                for (label, hr_num), variants in grouped.items()
            ]
            if not hr_group_pool:
                raise ValueError("No HR groups available for random concatenation.")
            if files_per_segment > len(hr_group_pool):
                raise ValueError("files_per_segment is larger than number of unique HR groups.")
        if not allow_replacement and len(labeled_files) < files_per_segment:
            raise ValueError("Not enough files for sampling without replacement.")

        target_seconds = target_minutes * 60.0
        created = 0
        attempts = 0
        max_attempts = max(n_outputs * 50, 200)
        stall_attempts = 0
        max_stall_attempts = max(400, n_outputs * 10)
        current_min_majority_ratio = float(min_majority_ratio)
        current_hr_band_padding = 0.0
        reject_stats = Counter()
        existing_hashes_cache: Dict[str, set] = {}

        while created < n_outputs and attempts < max_attempts:
            attempts += 1
            if adaptive_relaxation and stall_attempts >= max_stall_attempts:
                old_ratio = current_min_majority_ratio
                current_min_majority_ratio = max(0.55, current_min_majority_ratio - 0.02)
                current_hr_band_padding = min(3.0, current_hr_band_padding + 0.5)
                stall_attempts = 0
                print(
                    f"Adaptive relaxation -> min_majority_ratio: {old_ratio:.2f} -> {current_min_majority_ratio:.2f}, "
                    f"hr_band_padding: +/-{current_hr_band_padding:.1f}"
                )
            n_files_this_output = rng.randint(files_per_segment, files_per_segment_max)
            if choose_unique_hr_variants:
                n_pick = min(n_files_this_output, len(hr_group_pool))
                selected_groups = rng.sample(hr_group_pool, n_pick)
                segments = [
                    (label, rng.choice(variants))
                    for label, _, variants in selected_groups
                ]
            else:
                if not allow_replacement and len(labeled_files) < n_files_this_output:
                    raise ValueError("Not enough files for this output when sampling without replacement.")
                if allow_replacement:
                    segments = [rng.choice(labeled_files) for _ in range(n_files_this_output)]
                else:
                    segments = rng.sample(labeled_files, n_files_this_output)

            label_counts = Counter([label for label, _ in segments])
            majority_label = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
            majority_ratio = float(label_counts[majority_label] / max(len(segments), 1))
            if required_majority_label is not None and majority_label != required_majority_label:
                reject_stats['majority_label_mismatch'] += 1
                stall_attempts += 1
                continue
            if majority_ratio < current_min_majority_ratio:
                reject_stats['majority_ratio'] += 1
                stall_attempts += 1
                continue
            if hr_band_by_label:
                band = hr_band_by_label.get(majority_label)
                if band is not None:
                    hr_values_majority = [
                        self._extract_hr_number(filename)
                        for label, filename in segments
                        if label == majority_label
                    ]
                    if not hr_values_majority:
                        reject_stats['empty_majority_hr_values'] += 1
                        stall_attempts += 1
                        continue
                    mean_hr_majority = sum(hr_values_majority) / len(hr_values_majority)
                    low = band[0] - current_hr_band_padding
                    high = band[1] + current_hr_band_padding
                    if mean_hr_majority < low or mean_hr_majority > high:
                        reject_stats['hr_band'] += 1
                        stall_attempts += 1
                        continue

            output_df, _, _ = self._build_duration_segment(segments, target_seconds)
            if len(labels) > 1:
                final_dir = os.path.join(output_dir, f"mixed_{majority_label}")
                file_prefix = f"mixed_{majority_label}"
            else:
                final_dir = output_dir
                file_prefix = f"concat_{majority_label}"
            os.makedirs(final_dir, exist_ok=True)

            if final_dir not in existing_hashes_cache:
                existing_hashes_cache[final_dir] = self._collect_hashes_in_folder(final_dir)

            content_hash = self._hash_dataframe(output_df)
            if content_hash in existing_hashes_cache[final_dir]:
                reject_stats['duplicate_hash'] += 1
                stall_attempts += 1
                continue

            next_index = self._get_next_file_index(final_dir, file_prefix)
            file_name = f"{file_prefix}_{next_index:03d}.csv"
            output_path = os.path.join(final_dir, file_name)
            output_df.to_csv(output_path, index=False)

            existing_hashes_cache[final_dir].add(content_hash)
            created += 1
            stall_attempts = 0

        print(f"Random generation done: created={created}, attempts={attempts}, skipped={attempts - created}")
        if reject_stats:
            print(f"Reject stats: {dict(reject_stats)}")
        if created < n_outputs:
            print("Warning: unable to create all requested unique outputs with current constraints.")

    def _build_duration_segment(
        self,
        segments: List[Tuple[int, str]],
        target_seconds: float,
        *,
        rng: Optional[random.Random] = None,
        randomize_truncation: bool = False,
    ) -> Tuple[pd.DataFrame, int, float]:
        label_counts = Counter([label for label, _ in segments])
        majority_label = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
        majority_ratio = float(label_counts[majority_label] / max(len(segments), 1))

        output_parts = []
        total_duration = 0.0
        time_offset = 0.0

        for label, filename in segments:
            full_path = self._get_full_path(label, filename)
            df = self.data_cache[full_path]
            duration = self._get_duration_from_dataframe(df)
            remaining = target_seconds - total_duration
            if remaining <= 0:
                break
            if duration <= remaining:
                output_parts.append(self._offset_time(df, time_offset))
                total_duration += duration
                if "Time" in df.columns and len(df) > 0:
                    time_offset = output_parts[-1]["Time"].iloc[-1]
                continue

            truncated = df.copy()
            if "Time" in truncated.columns and len(truncated) > 0:
                start_time = float(truncated["Time"].iloc[0])
                end_time = float(truncated["Time"].iloc[-1])
                duration = end_time - start_time
                if randomize_truncation and rng is not None and duration > remaining:
                    max_shift = max(0.0, duration - remaining)
                    window_start = start_time + (rng.random() * max_shift)
                    window_end = window_start + remaining
                    truncated = truncated[(truncated["Time"] >= window_start) & (truncated["Time"] <= window_end)]
                    if len(truncated) == 0:
                        truncated = df[df["Time"] <= window_end].tail(1)
                    truncated = truncated.copy()
                    truncated["Time"] = truncated["Time"] - float(truncated["Time"].iloc[0])
                else:
                    cutoff = start_time + remaining
                    truncated = truncated[truncated["Time"] <= cutoff]
                    if len(truncated) > 0:
                        truncated = truncated.copy()
                        truncated["Time"] = truncated["Time"] - float(truncated["Time"].iloc[0])
            else:
                truncated = truncated.iloc[: int(remaining)]
            output_parts.append(self._offset_time(truncated, time_offset))
            total_duration = target_seconds
            break

        if not output_parts:
            raise ValueError("Unable to build concatenated segment from provided files.")

        return pd.concat(output_parts, ignore_index=True), majority_label, majority_ratio

## Usage examples

In [ ]:
from typing import Any, Iterable
import math
import os
import random
from collections import Counter


def _hhmm_to_seconds(hhmm: str) -> int:
    parts = hhmm.strip().split(':')
    if len(parts) != 2:
        raise ValueError(f"Time '{hhmm}' must be in HH:MM format.")
    h, m = int(parts[0]), int(parts[1])
    # Accept 24:00 as end-of-day boundary for full-day plans.
    if h == 24 and m == 0:
        return 24 * 3600
    if not (0 <= h <= 23 and 0 <= m <= 59):
        raise ValueError(f"Invalid time '{hhmm}'.")
    return h * 3600 + m * 60


def _seconds_to_hhmm(seconds: int) -> str:
    if seconds < 0 or seconds > 24 * 3600:
        raise ValueError('seconds must be in [0, 86400].')
    if seconds == 24 * 3600:
        return '24:00'
    h = seconds // 3600
    m = (seconds % 3600) // 60
    return f"{h:02d}:{m:02d}"


def build_day_plan(
    start: str = '00:00',
    end: str = '24:00',
    pattern: Any = None,
    segment_minutes: int = 60,
):
    start_sec = _hhmm_to_seconds(start)
    end_sec = _hhmm_to_seconds(end)
    if end_sec <= start_sec:
        raise ValueError('end must be after start.')
    if segment_minutes <= 0:
        raise ValueError('segment_minutes must be > 0.')
    step = segment_minutes * 60
    if (end_sec - start_sec) % step != 0:
        raise ValueError('Time range must be divisible by segment_minutes.')
    if not pattern:
        pattern = [
            ('pure', 0),
            ('pure', 0),
            ('mixed', 1),
            ('pure', 1),
            ('mixed', 2),
            ('pure', 2),
            ('mixed', 3),
            ('pure', 3),
            ('mixed', 2),
            ('pure', 2),
            ('mixed', 1),
            ('pure', 1),
            ('mixed', 0),
            ('pure', 0),
        ]
    for kind, level in pattern:
        if str(kind).lower() not in {'pure', 'mixed'}:
            raise ValueError(f"Invalid kind in pattern: {kind}")
        _ = int(level)
    n_segments = (end_sec - start_sec) // step
    day_plan = []
    for i in range(n_segments):
        seg_start = start_sec + i * step
        seg_end = seg_start + step
        kind, level = pattern[i % len(pattern)]
        day_plan.append((
            _seconds_to_hhmm(seg_start),
            _seconds_to_hhmm(seg_end),
            str(kind).lower(),
            int(level),
        ))
    return day_plan


def build_explicit_day_plan(
    segments: Iterable[Any],
    require_continuous_plan: bool = True,
):
    if not segments:
        raise ValueError('segments cannot be empty.')

    normalized = []
    for item in segments:
        if isinstance(item, dict):
            start = str(item['start'])
            end = str(item['end'])
            kind = str(item['kind']).lower()
            level = int(item['level'])
        elif isinstance(item, (list, tuple)) and len(item) == 4:
            start, end, kind, level = item
            start = str(start)
            end = str(end)
            kind = str(kind).lower()
            level = int(level)
        else:
            raise ValueError(f'Invalid segment item: {item}')

        if kind not in {'pure', 'mixed'}:
            raise ValueError(f"kind must be 'pure' or 'mixed', got: {kind}")

        start_sec = _hhmm_to_seconds(start)
        end_sec = _hhmm_to_seconds(end)
        if end_sec <= start_sec:
            raise ValueError(f'End time must be after start time: {item}')

        normalized.append({
            'start': start,
            'end': end,
            'start_sec': start_sec,
            'end_sec': end_sec,
            'kind': kind,
            'level': level,
        })

    normalized.sort(key=lambda x: x['start_sec'])

    for i in range(1, len(normalized)):
        prev = normalized[i - 1]
        cur = normalized[i]
        if cur['start_sec'] < prev['end_sec']:
            raise ValueError(
                f"Timeline overlaps between {prev['start']}-{prev['end']} and {cur['start']}-{cur['end']}."
            )
        if require_continuous_plan and cur['start_sec'] != prev['end_sec']:
            raise ValueError(
                f"Timeline is not continuous between {prev['end']} and {cur['start']}."
            )

    return [
        (item['start'], item['end'], item['kind'], item['level'])
        for item in normalized
    ]


def assign_label_column(df: pd.DataFrame, day_plan: list):
    """
    Gán cột label cho từng đoạn dựa trên day_plan.

    Lưu ý: Hàm này giả định `day_plan` liên tục (không có gap) và sẽ gán
    nhãn dựa trên độ dài từng segment theo thứ tự (không dùng absolute HH:MM).
    """
    segment_bounds = []
    offset = 0.0
    for seg in day_plan:
        if isinstance(seg, dict):
            duration = _hhmm_to_seconds(seg['end']) - _hhmm_to_seconds(seg['start'])
            label = int(seg['level'])
        else:
            duration = _hhmm_to_seconds(seg[1]) - _hhmm_to_seconds(seg[0])
            label = int(seg[3])
        segment_bounds.append((offset, offset + duration, label))
        offset += duration

    labels = []
    for t in df['Time']:
        found = False
        for start, end, label in segment_bounds:
            if start <= t < end or (abs(t - end) < 1e-6 and t == df['Time'].iloc[-1]):
                labels.append(label)
                found = True
                break
        if not found:
            labels.append(None)

    df = df.copy()
    df['label'] = labels
    return df


def concatenate_by_daily_scenario(
    *,
    day_plan: list,
    output_dir: str,
    output_prefix: str,
    raw_base_dir: str,
    random_seed: int | None = None,
    require_continuous_plan: bool = True,
    mixed_majority_ratio: float = 0.65,
    labels: list[int] | None = None,
    allow_reuse_files: bool = True,
    mixed_other_pick_prob: float = 0.45,
):
    """Tạo 1 file concat theo `day_plan` và đảm bảo KHÔNG có gap lớn.

    Mấu chốt để không có gap: mỗi segment phải tạo đủ `target_seconds`.
    Dataset raw hiện tại có tổng duration theo label nhỏ hơn 24h, nên phải cho
    phép tái sử dụng (reuse) file nguồn để ghép đủ độ dài.

    - `pure`: chọn file trong đúng `level` và lặp lại nếu thiếu duration.
    - `mixed`: chọn file nhiều label nhưng ép majority label = `level` với tỷ lệ >= `mixed_majority_ratio`,
      đồng thời lặp lại file nếu thiếu duration.
    """
    if not day_plan:
        raise ValueError('day_plan cannot be empty.')

    if mixed_majority_ratio < 0.5 or mixed_majority_ratio > 1.0:
        raise ValueError('mixed_majority_ratio must be in [0.5, 1.0].')

    if mixed_other_pick_prob < 0.0 or mixed_other_pick_prob > 1.0:
        raise ValueError('mixed_other_pick_prob must be in [0, 1].')

    rng = random.Random(random_seed)

    concat = ECGAdvancedConcatenator(
        csv_label_file=None,
        data_dir=raw_base_dir,
        labels=labels or [0, 1, 2, 3],
    )

    # Validate plan continuity to match assign_label_column() behavior.
    expected_start = _hhmm_to_seconds(day_plan[0][0])
    if require_continuous_plan and expected_start != 0:
        raise ValueError('When require_continuous_plan=True, plan should start at 00:00.')

    for i in range(1, len(day_plan)):
        prev = day_plan[i - 1]
        cur = day_plan[i]
        prev_end = _hhmm_to_seconds(prev[1])
        cur_start = _hhmm_to_seconds(cur[0])
        if require_continuous_plan and cur_start != prev_end:
            raise ValueError(
                f"Plan is not continuous between {prev[1]} and {cur[0]} (segment index {i})."
            )

    def _clean_columns(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.columns = [str(c).strip() for c in df.columns]
        unnamed = [c for c in df.columns if str(c).lower().startswith('unnamed')]
        if unnamed:
            df = df.drop(columns=unnamed)
        return df

    def _get_file_duration(label: int, filename: str) -> float:
        concat._load_label_files(label)
        full_path = concat._get_full_path(label, filename)
        df = concat.data_cache[full_path]
        return float(concat._get_duration_from_dataframe(df))

    def _segments_pure_exact_duration(label: int, target_seconds: float) -> list[tuple[int, str]]:
        concat._load_label_files(label)
        files = list(concat.label_files.get(label, []))
        if not files:
            raise ValueError(f'No files available for label={label}')

        segments: list[tuple[int, str]] = []
        total = 0.0
        safety = 0
        while total < target_seconds:
            safety += 1
            if safety > 20000:
                raise RuntimeError('Safety stop: too many iterations while building pure segment.')

            filename = rng.choice(files) if allow_reuse_files else files[len(segments) % len(files)]
            duration = _get_file_duration(label, filename)
            if duration <= 0:
                continue
            segments.append((label, filename))
            total += duration

        return segments

    def _segments_mixed_exact_duration(required_majority_label: int, target_seconds: float) -> list[tuple[int, str]]:
        labels_all = list(concat.labels)
        for lb in labels_all:
            concat._load_label_files(lb)

        pools: dict[int, list[str]] = {lb: list(concat.label_files.get(lb, [])) for lb in labels_all}
        if not pools.get(required_majority_label):
            raise ValueError(f'No files available for majority label={required_majority_label}')

        other_labels = [lb for lb in labels_all if lb != required_majority_label and pools.get(lb)]

        segments: list[tuple[int, str]] = []
        counts = Counter()
        total = 0.0
        safety = 0

        while total < target_seconds:
            safety += 1
            if safety > 40000:
                raise RuntimeError('Safety stop: too many iterations while building mixed segment.')

            n_total = len(segments)
            maj = counts[required_majority_label]

            # If picking a non-majority next would violate the ratio, force majority.
            if n_total == 0:
                pick_label = required_majority_label
            else:
                projected_ratio_if_other = maj / float(n_total + 1)
                if projected_ratio_if_other < mixed_majority_ratio:
                    pick_label = required_majority_label
                else:
                    if other_labels and rng.random() < mixed_other_pick_prob:
                        pick_label = rng.choice(other_labels)
                    else:
                        pick_label = required_majority_label

            filename = rng.choice(pools[pick_label])
            duration = _get_file_duration(pick_label, filename)
            if duration <= 0:
                continue

            segments.append((pick_label, filename))
            counts[pick_label] += 1
            total += duration

        # Final guarantee: majority ratio by file count.
        while counts[required_majority_label] / float(len(segments)) < mixed_majority_ratio:
            filename = rng.choice(pools[required_majority_label])
            duration = _get_file_duration(required_majority_label, filename)
            if duration <= 0:
                continue
            segments.append((required_majority_label, filename))
            counts[required_majority_label] += 1
            total += duration

        # Try to ensure it's truly mixed when possible.
        if other_labels and all(lb == required_majority_label for lb, _ in segments):
            # Replace one element with a non-majority file (ratio should still hold).
            swap_label = rng.choice(other_labels)
            swap_file = rng.choice(pools[swap_label])
            segments[-1] = (swap_label, swap_file)

        return segments

    offset_seconds = 0.0
    parts: list[pd.DataFrame] = []

    for seg in day_plan:
        start, end, kind, level = seg
        duration_sec = _hhmm_to_seconds(end) - _hhmm_to_seconds(start)
        if duration_sec <= 0:
            raise ValueError(f'Invalid segment duration: {seg}')

        kind = str(kind).lower()
        level = int(level)
        target_seconds = float(duration_sec)

        if kind == 'pure':
            segments = _segments_pure_exact_duration(level, target_seconds)
            seg_df, majority_label, _ = concat._build_duration_segment(segments, target_seconds)
            if majority_label != level:
                raise ValueError(f'Pure segment majority label mismatch: expected {level}, got {majority_label}')
        elif kind == 'mixed':
            segments = _segments_mixed_exact_duration(level, target_seconds)
            seg_df, majority_label, majority_ratio = concat._build_duration_segment(segments, target_seconds)
            if majority_label != level or majority_ratio < mixed_majority_ratio:
                raise ValueError(
                    f"Mixed segment check failed: expected majority={level} ratio>={mixed_majority_ratio}, "
                    f"got majority={majority_label}, ratio={majority_ratio:.3f}."
                )
        else:
            raise ValueError(f"Unknown kind '{kind}'. Expected 'pure' or 'mixed'.")

        seg_df = _clean_columns(seg_df)
        if 'Time' not in seg_df.columns:
            raise ValueError('Expected Time column in concatenated segment.')

        # Enforce exact segment span so day boundaries match plan and no gaps appear.
        seg_df = ECGAdvancedConcatenator._offset_time(seg_df, float(offset_seconds))
        parts.append(seg_df)
        offset_seconds += float(duration_sec)

    daily_df = pd.concat(parts, ignore_index=True)
    daily_df = _clean_columns(daily_df)
    daily_df = daily_df.sort_values('Time', kind='mergesort').reset_index(drop=True)

    os.makedirs(output_dir, exist_ok=True)
    next_index = ECGAdvancedConcatenator._get_next_file_index(output_dir, output_prefix)
    saved_path = os.path.join(output_dir, f"{output_prefix}_{next_index:03d}.csv")
    daily_df.to_csv(saved_path, index=False)

    print(f"Saved: {saved_path}")
    print(f"Segments: {len(day_plan)}, total_seconds={offset_seconds:.0f}, rows={len(daily_df)}")

    return daily_df, saved_path


# Option 1: explicit per-segment times (you can set 07:15 -> 08:15, etc.)
USE_EXPLICIT_SEGMENTS = True

EXPLICIT_SEGMENTS = [
    ('00:00', '05:00', 'pure', 0),
    ('05:00', '06:00', 'mixed', 0),
    ('06:00', '07:15', 'pure', 0),
    ('07:15', '08:15', 'mixed', 0),
    ('08:15', '08:30', 'mixed', 1),
    ('08:30', '09:00', 'mixed', 1),
    ('09:00', '10:00', 'pure', 2),
    ('10:00', '11:00', 'pure', 2),
    ('11:00', '11:30', 'pure', 3),
    ('11:30', '12:00', 'mixed', 1),
    ('12:00', '13:30', 'pure',0),
    ('13:30', '14:00', 'mixed', 2),
    ('14:00', '14:10', 'mixed', 0),
    ('14:10', '15:00', 'mixed', 3),
    ('15:00', '16:00', 'mixed', 3),
    ('16:00', '16:10', 'mixed', 2),
    ('16:10', '16:30', 'pure', 3),
    ('16:30', '17:30', 'pure', 3),
    ('17:30', '18:30', 'mixed', 2),
    ('18:30', '18:45', 'pure', 1),
    ('18:45', '18:50', 'pure', 0),
    ('18:50', '19:30', 'pure', 1),
    ('19:30', '20:00', 'mixed', 1),
    ('20:00', '21:00', 'pure', 1),
    ('21:00', '22:00', 'mixed', 0),
    ('22:00', '23:00', 'pure', 1),
    ('23:00', '24:00', 'pure', 0),
]

# Option 2: auto-build repeated pattern for a full day
START_TIME = '00:00'
END_TIME = '24:00'
SEGMENT_MINUTES = 60
PATTERN = [
    ('pure', 0),
    ('pure', 0),
    ('mixed', 1),
    ('pure', 1),
    ('mixed', 2),
    ('pure', 2),
    ('mixed', 3),
    ('pure', 3),
    ('mixed', 2),
    ('pure', 2),
    ('mixed', 1),
    ('pure', 1),
    ('mixed', 0),
    ('pure', 0),
]

if USE_EXPLICIT_SEGMENTS:
    DAY_PLAN = build_explicit_day_plan(
        segments=EXPLICIT_SEGMENTS,
        require_continuous_plan=True,
    )
    output_prefix = 'day_custom_segments'
else:
    DAY_PLAN = build_day_plan(
        start=START_TIME,
        end=END_TIME,
        pattern=PATTERN,
        segment_minutes=SEGMENT_MINUTES,
    )
    output_prefix = 'day_full_0000_2400'


# Tạo nhiều file trong 1 lần chạy (không cần run cell nhiều lần)
N_FILES = 15
BASE_SEED = None  # đặt số (vd 42) để reproducible; None = random
created_paths = []

for i in range(N_FILES):
    seed_i = None if BASE_SEED is None else int(BASE_SEED) + i

    daily_df, saved_path = concatenate_by_daily_scenario(
        day_plan=DAY_PLAN,
        output_dir='data/concatenated/day_scenarios',
        output_prefix=output_prefix,
        raw_base_dir='data/raw_gen',
        random_seed=seed_i,
        require_continuous_plan=True,
        mixed_majority_ratio=0.65,
    )

    # Thêm cột label cho từng dòng và lưu ra file (overwrite file vừa tạo)
    if daily_df is not None and len(daily_df) > 0:
        daily_df_labeled = assign_label_column(daily_df, DAY_PLAN)
        daily_df_labeled.to_csv(saved_path, index=False)

    created_paths.append(saved_path)

print('Done. Created paths (first 3):')
print(created_paths[:3])
print(f'Total created: {len(created_paths)}')

display(daily_df.head())
print(f'Last output file: {saved_path}')


KeyboardInterrupt: 

## Tạo dataset A (overlap 4%) từ `raw_gen` bằng cách concatenate

> Bạn đã có dữ liệu theo label trong `data/raw_gen/{0,1,2,3}`. Mục tiêu ở đây là tạo thêm **một bộ dataset mới** (lưu vào folder mới) sao cho **~4% thời lượng** đến từ các file có HR nằm trong *vùng chồng lặp* giữa các lớp theo bảng 4.1.

Vì các khoảng HR_mean trong bảng 4.1 chồng lặp rất rộng, ta **không** lấy toàn bộ phần giao nhau làm “overlap” (sẽ quá nhiều). Thay vào đó, ta làm đúng theo yêu cầu “4%” bằng cách:

- Xác định **HR overlap-range theo bảng 4.1** cho từng lớp (union giao nhau với lớp kề).
- Khi ghép file cho từng segment, giữ nguyên nhãn (vẫn lấy file trong đúng folder label), nhưng **chỉ chọn một tỷ lệ nhỏ** (ví dụ 4% theo thời lượng) các file có HR thuộc overlap-range.
- Các file còn lại chọn từ phần HR ngoài overlap-range.

Kết quả: phân phối HR_mean giữa các lớp sẽ có phần giao nhau (overlap) đúng theo tỷ lệ đặt, nhưng không cần sinh data mới.

Gợi ý: dataset B (không overlap) = đặt `target_overlap_ratio=0.0` và đổi `output_dir` sang một folder khác.

In [9]:
import os
import random
import re
from collections import Counter
from typing import Dict, List, Tuple

import pandas as pd
from IPython.display import display


# Bảng 4.1: khoảng HR_mean (bpm) theo nhãn (0..3)
HR_MEAN_RANGE_BY_LABEL: Dict[int, Tuple[int, int]] = {
    0: (74, 94),   # Bình thường
    1: (76, 96),   # Thấp
    2: (79, 100),  # Trung bình
    3: (80, 103),  # Cao
}


def _extract_hr_number_from_filename(filename: str) -> int:
    base = os.path.splitext(os.path.basename(filename))[0].lower()
    m = re.search(r"hr(\d+)", base)
    if m:
        return int(m.group(1))
    first_token = base.split("_", 1)[0]
    m2 = re.search(r"(\d+)", first_token)
    if m2:
        return int(m2.group(1))
    raise ValueError(f"Unable to extract HR number from filename: {filename}")


def _overlap_hr_set_by_label(hr_range_by_label: Dict[int, Tuple[int, int]]) -> Dict[int, set]:
    """Overlap HR-set cho từng label = union giao nhau với label kề (l-1, l+1)."""
    labels = sorted(hr_range_by_label.keys())
    out: Dict[int, set] = {lb: set() for lb in labels}
    for lb in labels:
        lo, hi = hr_range_by_label[lb]
        for nb in (lb - 1, lb + 1):
            if nb not in hr_range_by_label:
                continue
            lo2, hi2 = hr_range_by_label[nb]
            inter_lo = max(lo, lo2)
            inter_hi = min(hi, hi2)
            if inter_hi >= inter_lo:
                out[lb].update(range(int(inter_lo), int(inter_hi) + 1))
    return out


def concatenate_by_daily_scenario_overlap_control(
    *,
    day_plan: list,
    output_dir: str,
    output_prefix: str,
    raw_base_dir: str,
    target_overlap_ratio: float = 0.04,
    hr_range_by_label: Dict[int, Tuple[int, int]] = HR_MEAN_RANGE_BY_LABEL,
    random_seed: int | None = None,
    randomize_boundary_window: bool = True,
    require_continuous_plan: bool = True,
    mixed_majority_ratio: float = 0.65,
    labels: list[int] | None = None,
    allow_reuse_files: bool = True,
    mixed_other_pick_prob: float = 0.45,
    verbose: bool = True,
 ):
    """
    Concatenate từ `raw_base_dir` giống hàm cũ, nhưng điều khiển tỷ lệ overlap.

    Overlap (theo bảng 4.1):
    - Mỗi label có một HR overlap-set = union phần giao với label kề.
    - Một file coi là overlap nếu HR_number (trích từ tên file hrXX...) thuộc overlap-set của label đó.
    - `target_overlap_ratio` được đo theo *thời lượng (seconds)* ước lượng trong lúc chọn file.
    """
    if not day_plan:
        raise ValueError('day_plan cannot be empty.')
    if not (0.0 <= target_overlap_ratio <= 1.0):
        raise ValueError('target_overlap_ratio must be in [0, 1].')
    if mixed_majority_ratio < 0.5 or mixed_majority_ratio > 1.0:
        raise ValueError('mixed_majority_ratio must be in [0.5, 1.0].')
    if mixed_other_pick_prob < 0.0 or mixed_other_pick_prob > 1.0:
        raise ValueError('mixed_other_pick_prob must be in [0, 1].')

    # random_seed=None => different output each run (still reproducible if set)
    rng = random.Random(random_seed)

    concat = ECGAdvancedConcatenator(
        csv_label_file=None,
        data_dir=raw_base_dir,
        labels=labels or [0, 1, 2, 3],
    )

    # Validate plan continuity to match assign_label_column() behavior.
    if require_continuous_plan:
        expected_start = _hhmm_to_seconds(day_plan[0][0])
        if expected_start != 0:
            raise ValueError('When require_continuous_plan=True, plan should start at 00:00.')
        for i in range(1, len(day_plan)):
            prev = day_plan[i - 1]
            cur = day_plan[i]
            prev_end = _hhmm_to_seconds(prev[1])
            cur_start = _hhmm_to_seconds(cur[0])
            if cur_start != prev_end:
                raise ValueError(f"Plan is not continuous between {prev[1]} and {cur[0]} (segment index {i}).")

    overlap_set = _overlap_hr_set_by_label(hr_range_by_label)

    # Precompute pools per label: overlap vs non-overlap (by HR_number)
    pools: Dict[int, Dict[str, List[str]]] = {}
    for lb in concat.labels:
        concat._load_label_files(lb)
        files = list(concat.label_files.get(lb, []))
        overlap_files: List[str] = []
        nonoverlap_files: List[str] = []
        for fn in files:
            hr = _extract_hr_number_from_filename(fn)
            if hr in overlap_set.get(lb, set()):
                overlap_files.append(fn)
            else:
                nonoverlap_files.append(fn)
        pools[lb] = {'overlap': overlap_files, 'nonoverlap': nonoverlap_files, 'all': files}

    total_day_seconds = 0.0
    for start, end, _, _ in day_plan:
        total_day_seconds += float(_hhmm_to_seconds(end) - _hhmm_to_seconds(start))
    if total_day_seconds <= 0:
        raise ValueError('Total day duration must be > 0 seconds.')

    target_overlap_seconds = total_day_seconds * float(target_overlap_ratio)
    overlap_seconds_used = 0.0
    day_seconds_used = 0.0

    def _get_file_duration_seconds(label: int, filename: str) -> float:
        full_path = concat._get_full_path(label, filename)
        df = concat.data_cache[full_path]
        return float(concat._get_duration_from_dataframe(df))

    def _pick_filename(label: int, want_overlap: bool) -> str:
        p = pools[label]
        if want_overlap and p['overlap']:
            return rng.choice(p['overlap'])
        if p['nonoverlap']:
            return rng.choice(p['nonoverlap'])
        if p['overlap']:
            return rng.choice(p['overlap'])
        if p['all']:
            return rng.choice(p['all'])
        raise ValueError(f'No files available for label={label}')

    def _maybe_remove_if_no_reuse(label: int, filename: str) -> None:
        if allow_reuse_files:
            return
        for k in ('overlap', 'nonoverlap', 'all'):
            if filename in pools[label][k]:
                pools[label][k].remove(filename)

    def _segments_pure_exact_duration(label: int, target_seconds: float) -> List[Tuple[int, str]]:
        nonlocal overlap_seconds_used, day_seconds_used
        segments: List[Tuple[int, str]] = []
        total = 0.0
        safety = 0
        while total < target_seconds:
            safety += 1
            if safety > 20000:
                raise RuntimeError('Safety stop: too many iterations while building pure segment.')

            remaining = target_seconds - total
            remaining_budget = target_overlap_seconds - overlap_seconds_used
            remaining_day = max(total_day_seconds - day_seconds_used, 1e-9)
            p_overlap = max(0.0, min(1.0, remaining_budget / remaining_day))
            want_overlap = rng.random() < p_overlap

            filename = _pick_filename(label, want_overlap=want_overlap)
            duration = _get_file_duration_seconds(label, filename)
            if duration <= 0:
                continue

            contributed = min(duration, remaining)
            is_overlap = _extract_hr_number_from_filename(filename) in overlap_set.get(label, set())
            if is_overlap and target_overlap_seconds > 0:
                overlap_seconds_used = min(target_overlap_seconds, overlap_seconds_used + contributed)
            day_seconds_used = min(total_day_seconds, day_seconds_used + contributed)

            segments.append((label, filename))
            total += duration
            _maybe_remove_if_no_reuse(label, filename)

        return segments

    def _segments_mixed_exact_duration(required_majority_label: int, target_seconds: float) -> List[Tuple[int, str]]:
        nonlocal overlap_seconds_used, day_seconds_used
        labels_all = list(concat.labels)
        if required_majority_label not in labels_all:
            raise ValueError(f'Unknown majority label={required_majority_label}')

        other_labels = [lb for lb in labels_all if lb != required_majority_label and pools[lb]['all']]
        if not pools[required_majority_label]['all']:
            raise ValueError(f'No files available for majority label={required_majority_label}')

        segments: List[Tuple[int, str]] = []
        counts = Counter()
        total = 0.0
        safety = 0
        while total < target_seconds:
            safety += 1
            if safety > 40000:
                raise RuntimeError('Safety stop: too many iterations while building mixed segment.')

            n_total = len(segments)
            maj = counts[required_majority_label]

            if n_total == 0:
                pick_label = required_majority_label
            else:
                projected_ratio_if_other = maj / float(n_total + 1)
                if projected_ratio_if_other < mixed_majority_ratio:
                    pick_label = required_majority_label
                else:
                    if other_labels and rng.random() < mixed_other_pick_prob:
                        pick_label = rng.choice(other_labels)
                    else:
                        pick_label = required_majority_label

            remaining = target_seconds - total
            remaining_budget = target_overlap_seconds - overlap_seconds_used
            remaining_day = max(total_day_seconds - day_seconds_used, 1e-9)
            p_overlap = max(0.0, min(1.0, remaining_budget / remaining_day))
            want_overlap = rng.random() < p_overlap

            filename = _pick_filename(pick_label, want_overlap=want_overlap)
            duration = _get_file_duration_seconds(pick_label, filename)
            if duration <= 0:
                continue

            contributed = min(duration, remaining)
            is_overlap = _extract_hr_number_from_filename(filename) in overlap_set.get(pick_label, set())
            if is_overlap and target_overlap_seconds > 0:
                overlap_seconds_used = min(target_overlap_seconds, overlap_seconds_used + contributed)
            day_seconds_used = min(total_day_seconds, day_seconds_used + contributed)

            segments.append((pick_label, filename))
            counts[pick_label] += 1
            total += duration
            _maybe_remove_if_no_reuse(pick_label, filename)

        while counts[required_majority_label] / float(len(segments)) < mixed_majority_ratio:
            filename = _pick_filename(required_majority_label, want_overlap=False)
            duration = _get_file_duration_seconds(required_majority_label, filename)
            if duration <= 0:
                continue
            segments.append((required_majority_label, filename))
            counts[required_majority_label] += 1
            total += duration
            _maybe_remove_if_no_reuse(required_majority_label, filename)

        if other_labels and all(lb == required_majority_label for lb, _ in segments):
            swap_label = rng.choice(other_labels)
            swap_file = _pick_filename(swap_label, want_overlap=False)
            segments[-1] = (swap_label, swap_file)

        return segments

    def _clean_columns(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.columns = [str(c).strip() for c in df.columns]
        unnamed = [c for c in df.columns if str(c).lower().startswith('unnamed')]
        if unnamed:
            df = df.drop(columns=unnamed)
        return df

    offset_seconds = 0.0
    parts: List[pd.DataFrame] = []
    for start, end, kind, level in day_plan:
        duration_sec = _hhmm_to_seconds(end) - _hhmm_to_seconds(start)
        if duration_sec <= 0:
            raise ValueError(f'Invalid segment duration: {(start, end, kind, level)}')

        kind = str(kind).lower()
        level = int(level)
        target_seconds = float(duration_sec)

        if kind == 'pure':
            segments = _segments_pure_exact_duration(level, target_seconds)
            seg_df, majority_label, _ = concat._build_duration_segment(segments, target_seconds, rng=rng, randomize_truncation=randomize_boundary_window)
            if majority_label != level:
                raise ValueError(f'Pure segment majority label mismatch: expected {level}, got {majority_label}')
        elif kind == 'mixed':
            segments = _segments_mixed_exact_duration(level, target_seconds)
            seg_df, majority_label, majority_ratio = concat._build_duration_segment(segments, target_seconds, rng=rng, randomize_truncation=randomize_boundary_window)
            if majority_label != level or majority_ratio < mixed_majority_ratio:
                raise ValueError(
                    f"Mixed segment check failed: expected majority={level} ratio>={mixed_majority_ratio}, "
                    f"got majority={majority_label}, ratio={majority_ratio:.3f}."
                )
        else:
            raise ValueError(f"Unknown kind '{kind}'. Expected 'pure' or 'mixed'.")

        seg_df = _clean_columns(seg_df)
        if 'Time' not in seg_df.columns:
            raise ValueError('Expected Time column in concatenated segment.')

        seg_df = ECGAdvancedConcatenator._offset_time(seg_df, float(offset_seconds))
        parts.append(seg_df)
        offset_seconds += float(duration_sec)

    daily_df = pd.concat(parts, ignore_index=True)
    daily_df = _clean_columns(daily_df)
    daily_df = daily_df.sort_values('Time', kind='mergesort').reset_index(drop=True)

    os.makedirs(output_dir, exist_ok=True)
    next_index = ECGAdvancedConcatenator._get_next_file_index(output_dir, output_prefix)
    saved_path = os.path.join(output_dir, f"{output_prefix}_{next_index:03d}.csv")
    daily_df.to_csv(saved_path, index=False)

    achieved = overlap_seconds_used / total_day_seconds
    if verbose:
        print(f"Saved: {saved_path}")
        print(f"Segments: {len(day_plan)}, total_seconds={offset_seconds:.0f}, rows={len(daily_df)}")
        print(f"Target overlap ratio: {target_overlap_ratio:.4f} -> achieved~{achieved:.4f} (duration-estimate)")
        stats = {
            lb: {
                'overlap_files': len(pools[lb]['overlap']),
                'nonoverlap_files': len(pools[lb]['nonoverlap']),
            }
            for lb in sorted(pools)
        }
        display(pd.DataFrame(stats).T)

    return daily_df, saved_path

In [5]:
# Helper: generate multiple files for both generation modes
from __future__ import annotations

import os
from typing import Any, Optional


def _list_csv_files_recursive(folder: str) -> set[str]:
    out: set[str] = set()
    if not folder or not os.path.isdir(folder):
        return out
    for root, _, files in os.walk(folder):
        for name in files:
            if name.lower().endswith('.csv'):
                out.add(os.path.join(root, name))
    return out


def generate_ecg_files(
    *,
    kind: str,
    n_files: int,
    # Shared
    raw_base_dir: str = 'data/raw_gen',
    random_seed: int | None = None,
    verbose: bool = True,
    progress: bool = True,
    # DAY_PLAN mode
    day_plan: list | None = None,
    output_dir: str | None = None,
    output_prefix: str | None = None,
    target_overlap_ratio: float = 0.04,
    mixed_majority_ratio: float = 0.65,
    require_continuous_plan: bool = True,
    randomize_boundary_window: bool = True,
    add_label_column: bool = True,
    # RANDOM_HR mode
    labels: list[int] | None = None,
    files_per_segment: int = 8,
    files_per_segment_max: int | None = None,
    target_minutes: float = 30.0,
    allow_replacement: bool = True,
    required_majority_label: int | None = None,
    min_majority_ratio: float = 0.50,
    choose_unique_hr_variants: bool = True,
    adaptive_relaxation: bool = True,
    hr_band_by_label: dict[int, tuple[int, int]] | None = None,
 ) -> list[str]:
    """
    Gen nhiều file và tự tăng index theo file hiện có.

    kind:
      - 'day_plan': dùng concatenate_by_daily_scenario_overlap_control (Dataset A/B kiểu day scenario).
      - 'random_hr': dùng ECGAdvancedConcatenator.concatenate_random_hr.

    Returns: list các path file .csv mới tạo.
    """
    if n_files <= 0:
        raise ValueError('n_files must be > 0')

    kind = str(kind).strip().lower()
    if kind not in {'day_plan', 'random_hr'}:
        raise ValueError("kind must be 'day_plan' or 'random_hr'")

    if kind == 'day_plan':
        if day_plan is None:
            raise ValueError('day_plan is required for kind=day_plan')
        if not output_dir or not output_prefix:
            raise ValueError('output_dir and output_prefix are required for kind=day_plan')

        if add_label_column and 'assign_label_column' not in globals():
            raise RuntimeError("add_label_column=True but assign_label_column() is not defined in this notebook scope.")

        saved_paths: list[str] = []
        for i in range(int(n_files)):
            seed_i = None if random_seed is None else int(random_seed) + i
            daily_df, saved_path = concatenate_by_daily_scenario_overlap_control(
                day_plan=day_plan,
                output_dir=output_dir,
                output_prefix=output_prefix,
                raw_base_dir=raw_base_dir,
                target_overlap_ratio=float(target_overlap_ratio),
                random_seed=seed_i,
                require_continuous_plan=require_continuous_plan,
                mixed_majority_ratio=float(mixed_majority_ratio),
                randomize_boundary_window=randomize_boundary_window,
                # keep underlying logs quiet; we print progress ourselves
                verbose=False,
            )
            if add_label_column:
                daily_df = assign_label_column(daily_df, day_plan)
                daily_df.to_csv(saved_path, index=False)
            saved_paths.append(saved_path)
            if progress:
                print(f"[{i+1}/{n_files}] saved: {saved_path}")
        return saved_paths

    # kind == 'random_hr'
    if not output_dir:
        raise ValueError('output_dir is required for kind=random_hr')
    before = _list_csv_files_recursive(output_dir)

    concat = ECGAdvancedConcatenator(
        csv_label_file=None,
        data_dir=raw_base_dir,
        labels=labels or [0, 1, 2, 3],
    )
    concat.concatenate_random_hr(
        labels=labels or [0, 1, 2, 3],
        files_per_segment=int(files_per_segment),
        n_outputs=int(n_files),
        output_dir=output_dir,
        target_minutes=float(target_minutes),
        allow_replacement=bool(allow_replacement),
        random_seed=random_seed,
        files_per_segment_max=files_per_segment_max,
        required_majority_label=required_majority_label,
        min_majority_ratio=float(min_majority_ratio),
        hr_band_by_label=hr_band_by_label,
        choose_unique_hr_variants=bool(choose_unique_hr_variants),
        adaptive_relaxation=bool(adaptive_relaxation),
    )

    after = _list_csv_files_recursive(output_dir)
    created = sorted(list(after - before))
    if verbose:
        print(f'Created {len(created)} new csv files under: {output_dir}')
    return created


In [7]:
# Dataset A (overlap) - tự chứa DAY_PLAN
import pandas as pd
from IPython.display import display
from typing import Iterable, Any


def _hhmm_to_seconds(hhmm: str) -> int:
    parts = hhmm.strip().split(':')
    if len(parts) != 2:
        raise ValueError(f"Time '{hhmm}' must be in HH:MM format.")
    h, m = int(parts[0]), int(parts[1])
    if h == 24 and m == 0:
        return 24 * 3600
    if not (0 <= h <= 23 and 0 <= m <= 59):
        raise ValueError(f"Invalid time '{hhmm}'.")
    return h * 3600 + m * 60


def build_explicit_day_plan(
    segments: Iterable[Any],
    require_continuous_plan: bool = True,
 ):
    if not segments:
        raise ValueError('segments cannot be empty.')
    normalized = []
    for item in segments:
        if isinstance(item, (list, tuple)) and len(item) == 4:
            start, end, kind, level = item
            start = str(start)
            end = str(end)
            kind = str(kind).lower()
            level = int(level)
        else:
            raise ValueError(f'Invalid segment item: {item}')
        if kind not in {'pure', 'mixed'}:
            raise ValueError(f"kind must be 'pure' or 'mixed', got: {kind}")
        start_sec = _hhmm_to_seconds(start)
        end_sec = _hhmm_to_seconds(end)
        if end_sec <= start_sec:
            raise ValueError(f'End time must be after start time: {item}')
        normalized.append({
            'start': start,
            'end': end,
            'start_sec': start_sec,
            'end_sec': end_sec,
            'kind': kind,
            'level': level,
        })
    normalized.sort(key=lambda x: x['start_sec'])
    for i in range(1, len(normalized)):
        prev = normalized[i - 1]
        cur = normalized[i]
        if cur['start_sec'] < prev['end_sec']:
            raise ValueError(f"Timeline overlaps between {prev['start']}-{prev['end']} and {cur['start']}-{cur['end']}.")
        if require_continuous_plan and cur['start_sec'] != prev['end_sec']:
            raise ValueError(f"Timeline is not continuous between {prev['end']} and {cur['start']}.")
    return [(x['start'], x['end'], x['kind'], x['level']) for x in normalized]


def assign_label_column(df: pd.DataFrame, day_plan: list) -> pd.DataFrame:
    segment_bounds = []
    offset = 0.0
    for start, end, _, level in day_plan:
        duration = _hhmm_to_seconds(end) - _hhmm_to_seconds(start)
        label = int(level)
        segment_bounds.append((offset, offset + duration, label))
        offset += duration
    labels = []
    for t in df['Time']:
        for s, e, lb in segment_bounds:
            if s <= t < e or (abs(t - e) < 1e-6 and t == df['Time'].iloc[-1]):
                labels.append(lb)
                break
        else:
            labels.append(None)
    df = df.copy()
    df['label'] = labels
    return df


EXPLICIT_SEGMENTS = [
    ('00:00', '05:00', 'pure', 0),
    ('05:00', '06:00', 'mixed', 0),
    ('06:00', '07:15', 'pure', 0),
    ('07:15', '08:15', 'mixed', 0),
    ('08:15', '08:30', 'mixed', 1),
    ('08:30', '09:00', 'mixed', 1),
    ('09:00', '10:00', 'pure', 2),
    ('10:00', '11:00', 'pure', 2),
    ('11:00', '11:30', 'pure', 3),
    ('11:30', '12:00', 'mixed', 1),
    ('12:00', '13:30', 'pure', 0),
    ('13:30', '14:00', 'mixed', 2),
    ('14:00', '14:10', 'mixed', 0),
    ('14:10', '15:00', 'mixed', 3),
    ('15:00', '16:00', 'mixed', 3),
    ('16:00', '16:10', 'mixed', 2),
    ('16:10', '16:30', 'pure', 3),
    ('16:30', '17:30', 'pure', 3),
    ('17:30', '18:30', 'mixed', 2),
    ('18:30', '18:45', 'pure', 1),
    ('18:45', '18:50', 'pure', 0),
    ('18:50', '19:30', 'pure', 1),
    ('19:30', '20:00', 'mixed', 1),
    ('20:00', '21:00', 'pure', 1),
    ('21:00', '22:00', 'mixed', 0),
    ('22:00', '23:00', 'pure', 1),
    ('23:00', '24:00', 'pure', 0),
 ]

DAY_PLAN = build_explicit_day_plan(EXPLICIT_SEGMENTS, require_continuous_plan=True)

TARGET_OVERLAP_RATIO = 0.04  # tăng lên 0.06 / 0.08 nếu bạn muốn overlap nhiều hơn
OUTPUT_DIR_OVERLAP4 = 'data/concatenated/day_scenarios_overlap4'
OUTPUT_PREFIX_OVERLAP4 = 'day_custom_segments_ov4'

# Gen 1 file (như cũ):
# daily_df_A, saved_path_A = concatenate_by_daily_scenario_overlap_control(
#     day_plan=DAY_PLAN,
#     output_dir=OUTPUT_DIR_OVERLAP4,
#     output_prefix=OUTPUT_PREFIX_OVERLAP4,
#     raw_base_dir='data/raw_gen',
#     random_seed=None,
#     require_continuous_plan=True,
#     mixed_majority_ratio=0.65,
#     target_overlap_ratio=TARGET_OVERLAP_RATIO,
#  )

# Gen nhiều file (tự tăng tiếp index theo folder output):
# created_paths = generate_ecg_files(
#     kind='day_plan',
#     n_files=10,
#     day_plan=DAY_PLAN,
#     output_dir=OUTPUT_DIR_OVERLAP4,
#     output_prefix=OUTPUT_PREFIX_OVERLAP4,
#     raw_base_dir='data/raw_gen',
#     target_overlap_ratio=TARGET_OVERLAP_RATIO,
#     random_seed=None,
# )

# Ví dụ gen kiểu random_hr (files_per_segment=8, tạo 20 outputs):
# created_paths = generate_ecg_files(
#     kind='random_hr',
#     n_files=20,
#     output_dir='data/concatenated/random_hr',
#     raw_base_dir='data/raw_gen',
#     labels=[0, 1, 2, 3],
#     files_per_segment=8,
#     target_minutes=30,
#     random_seed=None,
# )

#if daily_df_A is not None and len(daily_df_A) > 0:
#    daily_df_A = assign_label_column(daily_df_A, DAY_PLAN)
#    daily_df_A.to_csv(saved_path_A, index=False)

#display(daily_df_A.head())
# print(f'Dataset A (overlap~{TARGET_OVERLAP_RATIO*100:.1f}%) saved: {saved_path_A}')

In [11]:
# Run: generate 10 random day_plan files with ~4% overlap
N_FILES = 15
# created_paths = generate_ecg_files(
#     kind='day_plan',
#     n_files=N_FILES,
#     day_plan=DAY_PLAN,
#     output_dir=OUTPUT_DIR_OVERLAP4,
#     output_prefix=OUTPUT_PREFIX_OVERLAP4,
#     raw_base_dir='data/raw_gen',
#     target_overlap_ratio=0.04,
#     random_seed=None,
#     randomize_boundary_window=True,
#     add_label_column=True,
#     progress=True,
#     verbose=False,
#  )
# Ví dụ gen kiểu random_hr (files_per_segment=8, tạo 20 outputs):
created_paths = generate_ecg_files(
    kind='random_hr',
    n_files=N_FILES,
    output_dir='data/concatenated/random_hr',
    raw_base_dir='data/raw_gen',
    labels=[0, 1, 2, 3],
    files_per_segment=8,
    target_minutes=30,
    random_seed=None,
)
print('Done. Created paths (first 3):')
print(created_paths[:3])
print(f'Total created: {len(created_paths)}')

Random generation done: created=15, attempts=33, skipped=18
Reject stats: {'majority_ratio': 18}
Created 15 new csv files under: data/concatenated/random_hr
Done. Created paths (first 3):
['data/concatenated/random_hr\\mixed_0\\mixed_0_001.csv', 'data/concatenated/random_hr\\mixed_0\\mixed_0_002.csv', 'data/concatenated/random_hr\\mixed_0\\mixed_0_003.csv']
Total created: 15
